# Database preparation

Create sql connection

In [1]:
import sqlite3

# connect to database (this creates gym.db if it does not exist)
conn = sqlite3.connect("gym.db")

# cursor to execute SQL
cur = conn.cursor()

Execute schema and data to make the database ready for use

In [5]:
# Execute schema.sql
with open("schema.sql", "r", encoding="utf-8") as f:  # Reads the file
    schema_sql = f.read()

cur.executescript(schema_sql)  # runs sql statements
conn.commit()  # saves changes

In [ ]:

# Execute data.sql
with open("data.sql", "r", encoding="utf-8") as f:
    data_sql = f.read()

cur.executescript(data_sql)
conn.commit()

# Query contens

Now we can query from the database

In [7]:
# Query the database
cur.execute("SELECT * FROM users")

rows = cur.fetchall()
for row in rows:
    print(row)

(1, 'Johnny', 'Nordmann', 'johnny@stud.ntnu.no', '40100001', 1)
(2, 'Viola', 'Smith', 'viola@stud.ntnu.no', '40100010', 1)
(3, 'Erik', 'Hansen', 'erik.hansen@stud.ntnu.no', '40100002', 1)
(4, 'Sara', 'Berg', 'sara.berg@stud.ntnu.no', '40100003', 1)
(5, 'Lucas', 'Johansen', 'lucas.johansen@stud.ntnu.no', '40100004', 0)
(6, 'Emma', 'Larsen', 'emma.larsen@stud.ntnu.no', '40100005', 1)
(7, 'Noah', 'Dahl', 'noah.dahl@stud.ntnu.no', '40100006', 0)
(8, 'Maja', 'Nilsen', 'maja.nilsen@stud.ntnu.no', '40100007', 1)
(9, 'Oliver', 'Strand', 'oliver.strand@stud.ntnu.no', '40100008', 0)
(10, 'Ingrid', 'Moen', 'ingrid.moen@stud.ntnu.no', '40100009', 1)


In [8]:
# Query the database
cur.execute("SELECT * FROM session_bookings")

rows = cur.fetchall()
for row in rows:
    print(row)

In [5]:
# Query the database
import pandas as pd
df = pd.read_sql_query("SELECT * FROM group_sessions", conn)

df

,id,activity_id,hall_id,instructor_id,start_time,end_time,signup_deadline,sessiondate
0,1,2,3,2,18:00,18:45,17:00,2026-03-16
1,2,3,6,8,16:00,16:45,15:00,2026-03-16
2,3,4,1,2,18:30,19:15,17:30,2026-03-17
3,4,1,5,8,17:00,17:45,16:00,2026-03-17
4,5,1,2,2,19:00,19:45,18:00,2026-03-18
5,6,3,4,8,16:00,16:45,15:00,2026-03-18


In [6]:
# See who is instructor
import pandas as pd


query = """
    SELECT 
        activities.name AS Activity, 
        users.first_name || ' ' || users.last_name AS Instructor, 
        group_sessions.sessiondate AS Date
    FROM users
    INNER JOIN group_sessions ON users.id = group_sessions.instructor_id
    INNER JOIN activities ON activities.id = group_sessions.activity_id
    """

df = pd.read_sql_query(query, conn)

df

,Activity,Instructor,Date
0,Spin 8x3min,Viola Smith,2026-03-16
1,Spin45,Maja Nilsen,2026-03-16
2,Spin60,Viola Smith,2026-03-17
3,Spin 4x4,Maja Nilsen,2026-03-17
4,Spin 4x4,Viola Smith,2026-03-18
5,Spin45,Maja Nilsen,2026-03-18


# Use cases
Options
* Print use cases, choose from list which to test
* Write how to perform each use case in the tutorial, let the terminal application be like a regular user log in


Tenker at hvis jeg "logger inn" som Johnny med brukernavnet. Her får jeg valgene:
* **Use case 1**: er data.sql filen
* **Use case 4**: Se ukentlig schedule for alle treningsøkter (use case 4). Dette er en query som er kodet inn slik at om sensor velger å se dette vil det printes ut pent. Skal sorteres i tid (merge alle gym), og der start dag og uke er parametre som settes før spørringen kjøres.
* **Use case 5**: Se personlig besøkshistorie for bruker (use case 5). Dette er en query som er kodet inn slik at om sensor velger å se dette vil det printes ut pent.
* **Use case 2**: Booke den spesifikke treningen (HUSK å se at den finnes før booking) (use case 2)
* **Use case 3**: Få spørsmål om hvor mange minutter før du ankom timen, sensor kan velge 4 minutter og får da opp at bruker ikke har møtt opp i tide og at en dot registreres (use case 3)
* **Use case 6**: Valg om å se blacklisting utført. Implementerer en sjekk av hvor mange dotter Johnny har, og sikrer at han kommer for sent gjenværende for å få tre dotter (tilfelle sensor har gitt en dott alt i forrige sjekk). Deretter blir Johnny svartelistet, bruker den check_30_dots funksjonen.
* **Use case 7**: Et valg om å se "topplista". Query som kjøres, printer ut de som har trent mest en gitt uke. Kan da inkludere noe data på at noen spesifikke brukere (for å hindre problem med ID) har booket noen timer. 
* **Use case 8**: Researchers choice, sql query som kodes inn og kjøres når man velger dette. 

# Interface

We can use Python as application layer to receive user input, call SQL queries, and show results, i.e. an interface between user and database.

## Time things

For example getting timestamps, checking if attended 5 min before deadline

In [18]:
from datetime import datetime
booking_time = datetime.now()
datetime_object = str((datetime.now()).replace(microsecond=0))
date_part = booking_time.date()
time_part = (booking_time.time()).replace(microsecond=0)
# time_part = (booking_time.time()).replace(second=0, microsecond=0)
print(booking_time)
print(str(date_part))
print(time_part)
print(datetime_object)
print(type(datetime_object))


2026-03-16 21:03:28.832294
2026-03-16
21:03:28
2026-03-16 21:03:28
<class 'str'>


In [19]:
# Modify things with timestamps to check for example attendance

from datetime import datetime, timedelta

str_datetime = str((datetime.now()).replace(microsecond=0))  # Datetime now in string
print(type(str_datetime))
print(str_datetime)

dt = datetime.strptime(str_datetime, "%Y-%m-%d %H:%M:%S")  # String to datetime
print(type(dt))
print(dt)

dt_new = dt - timedelta(minutes=5) # subtract 5 minutes

dt_str = dt_new.strftime("%Y-%m-%d %H:%M:%S")
print(type(dt_str))
print(dt_str)

<class 'str'>
2026-03-16 21:03:31
<class 'datetime.datetime'>
2026-03-16 21:03:31
<class 'str'>
2026-03-16 20:58:31


## Book a session

First: User login

In [11]:
import pandas as pd
df = pd.read_sql_query("""SELECT * FROM users""", conn)
df

,id,first_name,last_name,email,phone_number,student
0,1,Johnny,Nordmann,johnny@stud.ntnu.no,40100001,1
1,2,Viola,Smith,viola@stud.ntnu.no,40100010,1
2,3,Erik,Hansen,erik.hansen@stud.ntnu.no,40100002,1
3,4,Sara,Berg,sara.berg@stud.ntnu.no,40100003,1
4,5,Lucas,Johansen,lucas.johansen@stud.ntnu.no,40100004,0
5,6,Emma,Larsen,emma.larsen@stud.ntnu.no,40100005,1
6,7,Noah,Dahl,noah.dahl@stud.ntnu.no,40100006,0
7,8,Maja,Nilsen,maja.nilsen@stud.ntnu.no,40100007,1
8,9,Oliver,Strand,oliver.strand@stud.ntnu.no,40100008,0
9,10,Ingrid,Moen,ingrid.moen@stud.ntnu.no,40100009,1


Plan: User inserts their username (email), which is unique in the system. This email corresponds to a unique ID that is returned. This ID is further used in the booking. I do this because it simplifies stuff since user_id is used elsewhere

In [13]:
# Write in username (email), this is unique in the system so it corresponds to a unique user_id

email = str(input("Please enter username to log in: "))

cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
email_to_id = cur.fetchone()[0]

print(f"Welcome, {email}! user_id: {email_to_id}")

Welcome, maja.nilsen@stud.ntnu.no! user_id: 8


### Show available

In [ ]:
import pandas as pd
query = """SELECT 
                g.gym_name AS Gym,
                a.name,
                a.description,
                s.sessiondate AS Day,
                s.start_time AS Time
                --s.signup_deadline
            FROM group_sessions s
            INNER JOIN activities a ON a.id = s.activity_id
            INNER JOIN halls h ON h.id = s.hall_id
            INNER JOIN gyms g ON g.id = h.gym_id"""

df = pd.read_sql_query(query, conn)

df



,Gym,name,description,Day,Time
0,Øya,Spin 8x3min,Intervalltime på sykkel der du jobber i interv...,2026-03-16,18:00
1,Dragvoll,Spin45,Spinningtime med variert løype fordelt på 2-3 ...,2026-03-16,16:00
2,Øya,Spin60,Spinningtime med variert løype fordelt på 2-4 ...,2026-03-17,18:30
3,Dragvoll,Spin 4x4,"Intervalltime på sykkel med god oppvarming, de...",2026-03-17,17:00
4,Øya,Spin 4x4,"Intervalltime på sykkel med god oppvarming, de...",2026-03-18,19:00
5,Dragvoll,Spin45,Spinningtime med variert løype fordelt på 2-3 ...,2026-03-18,16:00


# Show available

In [17]:
# Show available, but in function that takes argument time

# AND group_sessions.sessiondate >= ?

def show_classes(week_num, start_day):
    ''' Function that prints classes'''

    week_num = None #fix later

    cur.execute(
        """SELECT 
                g.gym_name AS Gym,
                a.name,
                a.description,
                s.sessiondate AS Day,
                s.start_time AS Time
                --s.signup_deadline
            FROM group_sessions s
            INNER JOIN activities a ON a.id = s.activity_id
            INNER JOIN halls h ON h.id = s.hall_id
            INNER JOIN gyms g ON g.id = h.gym_id
            WHERE s.sessiondate > ?
            -- AND s.week = ?
            ORDER BY s.sessiondate, s.start_time
            ;
            """, (start_day,)
            )
    
    schedule = cur.fetchall()

    rows = []

    for s in schedule:
        row = [s[i] for i in range(len(s))]
        rows.append(row)

    # print(rows)
    for row in rows:
        print(f"==============================\n{row[3]} {row[4]}\n    {row[0]}\n    {row[1]}\n------------------------------\n    {row[2]}\n==============================\n\n\n")

    

show_classes(1, '2026-03-16')




2026-03-17 17:00
    Dragvoll
    Spin 4x4
------------------------------
    Intervalltime på sykkel med god oppvarming, deretter 4 stående intervaller på 4 minutter med ca 2 minutter aktiv pause mellom hvert drag. Timen avsluttes med nedsykling.



2026-03-17 18:30
    Øya
    Spin60
------------------------------
    Spinningtime med variert løype fordelt på 2-4 arbeidsperioder. Litt raskere tempo og lengre stående perioder enn Spin45, men du bestemmer selv intensiteten med motstanden du legger på.



2026-03-18 16:00
    Dragvoll
    Spin45
------------------------------
    Spinningtime med variert løype fordelt på 2-3 arbeidsperioder som passer for alle. Du bestemmer selv intensiteten emd hvor mye motstand du legger på.



2026-03-18 19:00
    Øya
    Spin 4x4
------------------------------
    Intervalltime på sykkel med god oppvarming, deretter 4 stående intervaller på 4 minutter med ca 2 minutter aktiv pause mellom hvert drag. Timen avsluttes med nedsykling.





User has to choose which class to go to, this give session_id. Since this will only be in CLI, I show numbered options and let user choose a number, since typing correct input can be annoying for user.

In [63]:
cur.execute("""SELECT distinct sessiondate FROM group_sessions ORDER BY sessiondate""")

dates = cur.fetchall()

print("Please select date by entering the number on the left:")
for i, d in enumerate(dates):
    print(i + 1, d[0])  # count from 1

date = int(input("Choose date: "))
selected_date = dates[date - 1][0]  # To take only the date string from a list of tuples on the form [(), ()], counting from zero

print("Selected date: ", selected_date)

Please select date by entering the number on the left:
1 2026-03-16
2 2026-03-17
3 2026-03-18
Selected date:  2026-03-16


In [64]:
df = pd.read_sql_query("""SELECT name, start_time, description, signup_deadline FROM activities INNER JOIN group_sessions ON activities.id = group_sessions.activity_id WHERE sessiondate='2026-03-17' ORDER BY group_sessions.start_time""", conn)

df

,name,start_time,description,signup_deadline
0,Spin 4x4,17:00,"Intervalltime på sykkel med god oppvarming, de...",16:00
1,Spin60,18:30,Spinningtime med variert løype fordelt på 2-4 ...,17:30


In [65]:
cur.execute("""SELECT g.id, a.name, g.start_time, a.description, g.signup_deadline FROM activities a INNER JOIN group_sessions g ON a.id = g.activity_id WHERE sessiondate=? ORDER BY g.start_time""", (selected_date,))

trainings = cur.fetchall()

print("Please select session by entering the number on the left:")
for i, (session_id, name, start_time, description, signup_deadline) in enumerate(trainings, start=1):
    print(f"{i}. {name}, starting {start_time}, signup deadline {signup_deadline}:")
    print(f"    {description}")
    print("     ")

training = int(input("Choose date: "))
selected_training = trainings[training - 1]  # To take only the training string from a list of tuples on the form [(), ()], counting from zero

print("Selected training: ", selected_training)

session_id, name, start_time, description, signup_deadline = selected_training

print(name)
print(session_id)

Please select session by entering the number on the left:
1. Spin45, starting 16:00, signup deadline 15:00:
    Spinningtime med variert løype fordelt på 2-3 arbeidsperioder som passer for alle. Du bestemmer selv intensiteten emd hvor mye motstand du legger på.
     
2. Spin 8x3min, starting 18:00, signup deadline 17:00:
    Intervalltime på sykkel der du jobber i intervaller på 3 minutter og kjører annethvert drag stående og sittende. Totalt 8 intervaller, med 1.5-2 min pause mellom. Oppvarming og nedtrapping er inkludert i timen.
     
Selected training:  (2, 'Spin45', '16:00', 'Spinningtime med variert løype fordelt på 2-3 arbeidsperioder som passer for alle. Du bestemmer selv intensiteten emd hvor mye motstand du legger på.', '15:00')
Spin45
2


In [22]:
#### Book session
from datetime import datetime


def book_session(user_id, session_id):
    '''
    Function that performs a booking
    --------------------
    user_id : Users unique id
    session_id: Sessions unique id (is retrieved from ****)
    '''

    #### First check if available spots by comparing booked spots with capacity
    cur.execute("""SELECT COUNT(*) FROM session_bookings WHERE session_id = ?;""", (session_id,))
    booked = cur.fetchone()[0]

    cur.execute("""SELECT capacity FROM halls JOIN group_sessions ON halls.id = group_sessions.hall_id WHERE group_sessions.id = ?;""", (session_id,))

    capacity = cur.fetchone()[0]

    if booked >= capacity:
        print("Session is full")
        return
    
    # TODO: Must register before booking deadline

    # If did not return above, means it was available spots so booking is performed:
    booking_time = str((datetime.now()).replace(microsecond=0))  # Booking time in format YYY-MM-DD HH:MM:SS

    cur.execute("""INSERT INTO session_bookings (session_id, user_id, booking_time) VALUES (?, ?, ?);""", (session_id, user_id, booking_time))
    booking_id = cur.lastrowid  # the autogenerated PK for that booking

    conn.commit()

    print("Booking successful")
    return booking_id

In [67]:
### User booking

print(type(email_to_id))
print(type(session_id))
booking_id = book_session(email_to_id, session_id)

<class 'int'>
<class 'int'>
Booking successful


## Create dot

### Check if arrived 5 min before

In [ ]:
from datetime import datetime, timedelta

def arrived_in_time(arrival_time, session_time):
    
    '''
    Function that checks if time1 is no less than 5 minutes before time2, and converts between string and datetime

    Returns True if user arrived on time, False if not
    '''

    # Make the times, which is in string, into datetime objects
    arrival_dt = datetime.strptime(arrival_time, "%Y-%m-%d %H:%M:%S") 
    session_dt = datetime.strptime(session_time, "%Y-%m-%d %H:%M:%S")
    # dt_new = dt - timedelta(minutes=5) # subtract 5 minutes
    # return dt_new.strftime("%Y-%m-%d %H:%M:%S")  # Return 5 minutes before

    five_min_before = session_dt - timedelta(minutes=5)

    print(f"Start time was {session_dt}\nDeadline for showing up is {five_min_before}.\nUser arrived at {arrival_dt}")

    if arrival_dt <= five_min_before:
        print("User arrived in time!")
        return 1  # user arrived in time
    else:
        print("User did not arrive in time.")
        return 0


### Give dot

In [ ]:
### Test
# session_time_test = str((datetime.now()).replace(microsecond=0))  # Datetime now in string
# dt = datetime.strptime(session_time_test, "%Y-%m-%d %H:%M:%S")  # String to datetime

diff = -4

# Apply difference
session_time_test = (datetime.now()).replace(microsecond=0)  # Datetime now 
# Make another that is an arrival with some difference
dt_new = session_time_test + timedelta(minutes=diff) # apply change

# Make strings
arrival_time_test = dt_new.strftime("%Y-%m-%d %H:%M:%S")
session_time_test = session_time_test.strftime("%Y-%m-%d %H:%M:%S")



arrived_in_time(arrival_time_test, session_time_test)

In [ ]:
# username = 'johnny@stud.ntnu.no'
user_id = email_to_id

# Johnny arrives to gym

diff = -4
session_time = (datetime.now()).replace(microsecond=0)  # Datetime now 
arrived = session_time + timedelta(minutes=diff) # apply change

# Make strings
arrival_time = arrived.strftime("%Y-%m-%d %H:%M:%S")
session_time = session_time.strftime("%Y-%m-%d %H:%M:%S")

print(arrival_time)
print(session_time)


def dot_giver(user_id, booking_id, arrival_time, session_time):
    '''Function that gives a dot to user if user did not arrive in time'''
    if arrived_in_time(arrival_time, session_time):
        # True if arrived on time
        # print("All good!")
        return
    else:
        # Make date for the dot
        today = datetime.now()
        date_part = booking_time.date()
        today_str = str(date_part)
        print(f"A dot is created for {today}")

        # Insert dot in system
        cur.execute("""INSERT INTO dot_system (user_id, session_booking_id, date_given) VALUES (?, ?, ?)""", (user_id, booking_id, today_str))

        conn.commit()

dot_giver(user_id, booking_id, arrival_time, session_time)

2026-03-16 21:00:51
2026-03-16 21:04:51


In [3]:
import pandas as pd
df = pd.read_sql_query("""SELECT * FROM dot_system""", conn)
df

,id,user_id,session_booking_id,date_given
0,1,1,3,2026-03-18
1,2,6,5,2026-03-18


### If three dots, ban user

In [ ]:
# Checks how many dots a user has. If over three, tell user she is banned, and when the first dot given expires. To be integrated with rest so that dot check happens before booking follows through. User never can get above three dots, since booking is not allowed if already has three books. But need a code to delete a dot after 30 days. Since I am not planning to have this updated everyday, instead, I will integrate into this a check if a dot can be deleted first, THEN this check



cur.execute("""SELECT COUNT(*) FROM dot_system WHERE user_id=?;""", (user_id,))
counts = cur.fetchone()[0]
print(counts)


# if counts >= 3:
if counts >= 1:
    # TODO Start by checking if a dot can be deleted (over 30 days old). If yes, delete before the rest of this code
    #  
    cur.execute("""SELECT date_given FROM dot_system WHERE user_id=? ORDER BY date_given ASC LIMIT 1;""", (user_id,))  # should order properly
    date_given = cur.fetchone()[0]

    date_given_dt = datetime.strptime(date_given, "%Y-%m-%d")  # Date dot was given
    dot_expires = date_given_dt + timedelta(days=30)  # Date dot expires
    dot_expires = dot_expires.date()
    today = datetime.now()  # Today
    today = today.date()

    print(date_given)
    print(dot_expires)
    print(today)
    
    print(counts)
    print(f"User {email_to_id} have been banned due to too many dots in the system. Booking is allowed again from the first dot expires, which is {dot_expires}")
    # This check happen at the start of booking, and

df = pd.read_sql_query("""SELECT * FROM dot_system WHERE user_id=?""", conn, params=(user_id,))
df

1
2026-03-16
2026-04-15
2026-03-16
1
User 4 have been banned due to too many dots in the system. Booking is allowed again from the first dot expires, which is 2026-04-15


,id,user_id,session_booking_id,date_given
0,1,4,1,2026-03-16


In [ ]:
# Option where we just check if 30 dots within 30 days, then we don't have to delete stuff
def check_dots_30_days(user_id):
    '''Checks how many dots were given the last 30 days'''
    thirtydays = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
    cur.execute("""SELECT COUNT(*) FROM dot_system WHERE user_id=? AND date_given >= ?;""", (user_id, thirtydays))
    active_dots = cur.fetchone()[0]
    if active_dots >= 3:
        print("User is blacklisted.")



def dot_count(user_id):
    '''Function that counts number of dots'''
    cur.execute("""SELECT COUNT(*) FROM dot_system WHERE user_id=?;""", (user_id,))
    counts = cur.fetchone()[0]
    return counts


def dot_expire(user_id):
    '''Function that returns oldest dot, useful to tell user when it expires'''
    cur.execute("""SELECT date_given FROM dot_system WHERE user_id=? ORDER BY date_given ASC LIMIT 1;""", (user_id,))  # Selects date of oldest dot
    date_given = cur.fetchone()[0] 

    date_given_dt = datetime.strptime(date_given, "%Y-%m-%d")  # Date dot was given
    dot_expires = date_given_dt + timedelta(days=30)  # Date dot expires
    dot_expires = dot_expires.date()
    today = datetime.now()  # Today
    today = today.date()


# def delete_expired_dots(user_id):
#     cur.execute("""SELECT date_given FROM dot_system WHERE user_id=? ORDER BY date_given ASC LIMIT 1;""", (user_id,))  # should order properly
#     date_given = cur.fetchone()[0]

#     date_given_dt = datetime.strptime(date_given, "%Y-%m-%d")  # Date dot was given
#     dot_expires = date_given_dt + timedelta(days=30)  # Date dot expires
#     dot_expires = dot_expires.date()
#     today = datetime.now()  # Today
#     today = today.date()

#     # if today > dot_expires:
#     #     cur.execute("""DELETE FROM dot_system""")



### Booking example

In [27]:
users = ['johnny@stud.ntnu.no', 'viola@stud.ntnu.no', 'erik.hansen@stud.ntnu.no', 'sara.berg@stud.ntnu.no']

for u in users:
    # Write in username (email), this is unique in the system so it corresponds to a unique user_id

    email = u

    cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
    email_to_id = cur.fetchone()[0]

    print(f"Welcome, {email}! user_id: {email_to_id}")


    ##### Choose date

    cur.execute("""SELECT distinct sessiondate FROM group_sessions ORDER BY sessiondate""")

    dates = cur.fetchall()

    print("Please select date by entering the number on the left:")
    for i, d in enumerate(dates):
        print(i + 1, d[0])  # count from 1

    date = int(input("Choose date: "))
    selected_date = dates[date - 1][0]  # To take only the date string from a list of tuples on the form [(), ()], counting from zero

    print("Selected date: ", selected_date)

    ### Chose training

    cur.execute("""SELECT g.id, a.name, g.start_time, a.description, g.signup_deadline FROM activities a INNER JOIN group_sessions g ON a.id = g.activity_id WHERE sessiondate=? ORDER BY g.start_time""", (selected_date,))

    trainings = cur.fetchall()

    print("Please select session by entering the number on the left:")
    for i, (session_id, name, start_time, description, signup_deadline) in enumerate(trainings, start=1):
        print(f"{i}. {name}, starting {start_time}, signup deadline {signup_deadline}:")
        print(f"    {description}")
        print("     ")

    training = int(input("Choose date: "))
    selected_training = trainings[training - 1]  # To take only the training string from a list of tuples on the form [(), ()], counting from zero

    print("Selected training: ", selected_training)

    session_id, name, start_time, description, signup_deadline = selected_training

    print(name)
    print(session_id)

    booking_id = book_session(email_to_id, session_id)

    diff = -4

    # Apply difference
    session_time_test = (datetime.now()).replace(microsecond=0)  # Datetime now 
    # Make another that is an arrival with some difference
    dt_new = session_time_test + timedelta(minutes=diff) # apply change

    # Make strings
    arrival_time_test = dt_new.strftime("%Y-%m-%d %H:%M:%S")
    session_time_test = session_time_test.strftime("%Y-%m-%d %H:%M:%S")



    arrived_in_time(arrival_time_test, session_time_test)
    dot_giver(user_id, booking_id, arrival_time, session_time)





        


print(type(email_to_id))
print(type(session_id))
# book_session(email_to_id, session_id)

Welcome, johnny@stud.ntnu.no! user_id: 1
Please select date by entering the number on the left:
1 2026-03-16
2 2026-03-17
3 2026-03-18
Selected date:  2026-03-17
Please select session by entering the number on the left:
1. Spin 4x4, starting 17:00, signup deadline 16:00:
    Intervalltime på sykkel med god oppvarming, deretter 4 stående intervaller på 4 minutter med ca 2 minutter aktiv pause mellom hvert drag. Timen avsluttes med nedsykling.
     
2. Spin60, starting 18:30, signup deadline 17:30:
    Spinningtime med variert løype fordelt på 2-4 arbeidsperioder. Litt raskere tempo og lengre stående perioder enn Spin45, men du bestemmer selv intensiteten med motstanden du legger på.
     
Selected training:  (3, 'Spin60', '18:30', 'Spinningtime med variert løype fordelt på 2-4 arbeidsperioder. Litt raskere tempo og lengre stående perioder enn Spin45, men du bestemmer selv intensiteten med motstanden du legger på.', '17:30')
Spin60
3
Booking successful
Start time was 2026-03-16 21:07:08


#### Email to ID

In [17]:
# Write in username (email), this is unique in the system so it corresponds to a unique user_id

email = 'johnny@stud.ntnu.no'

cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
email_to_id = cur.fetchone()[0]

print(email_to_id)

1


In [ ]:
#### Register when user doesn't show up

# I need some way to set the time so that it checks this 5 min before each specific group session. Could do it so that instructor checks attendance and register (1/0), but the task might be that the system should check this using the time registered, so I make some different attempts

# Register attendance
timestamp = str((datetime.now()).replace(microsecond=0))  # Might do it so that we set time before
attended = 1

def attendance_registration(session_id, timestamp, attended):
    ''' Function that checks if user attended session in time, and gives dot if not
    
    Parameters
    session_id  :    Already stored in variable earlier in booking stage
    attended    :      (boolean)
                        1 if user attended, 0 if not
    
    '''

    # Specific booking of a group session and its start time
    cur.execute("""SELECT 
                        g.start_time,
                        b.id
                FROM group_sessions g 
                INNER JOIN session_bookings b ON g.id = s.session_id 
                WHERE b.id=?""", (session_id,))
    
    
    showups = cur.fetchall()  # tuple with start_time, id for booking

    start_time, booking_id = showups[0]  # take out necessary info to register attendance
    fivemin_rule = five_min_before(start_time)

    
    # TODO to compare times, might actually keep both as datetime objects???

    

    # Register attendance
    # timestamp = str((datetime.now()).replace(microsecond=0))
    # attended = attended

    cur.execute("""INSERT INTO attendances (session_booking_id, timestamp, attended) VALUES (?, ?, ?)""", (session_id, timestamp, attended))


    






## Personal visit history

In [49]:
# Query the database
import pandas as pd
df = pd.read_sql_query("SELECT * FROM session_bookings", conn)

df

,id,session_id,user_id,booking_time
0,1,4,1,2026-03-18 12:13:56
1,2,1,1,2026-03-18 12:14:08
2,3,3,1,2026-03-18 12:14:18


In [59]:
import pandas as pd
# ASSUMPTION: Doesn't state if it should include attended or not. I am thinking that I can either add a column for this or not, I start with not.

# TODO Must be able to take start date as well

user_id = 1
email_to_id = user_id
from datetime import datetime
booking_time = datetime.now()
datetime_object = str((datetime.now()).replace(microsecond=0))
date_part = booking_time.date()

query = """
    SELECT 
        activities.name, 
        gyms.gym_name, 
        group_sessions.sessiondate, 
        group_sessions.start_time
    FROM group_sessions
    INNER JOIN activities ON group_sessions.activity_id = activities.id
    INNER JOIN halls ON group_sessions.hall_id = halls.id
    INNER JOIN gyms ON halls.gym_id = gyms.id
    INNER JOIN session_bookings ON session_bookings.session_id = group_sessions.id

    -- One user cannot book the same group session twice, so it should be unique, but check
    WHERE session_bookings.user_id = 1;

    """
# df = pd.read_sql_query(query, conn)

# df

cur.execute("""
    SELECT 
        activities.name, 
        gyms.gym_name, 
        group_sessions.sessiondate, 
        group_sessions.start_time
    FROM group_sessions
    INNER JOIN activities ON group_sessions.activity_id = activities.id
    INNER JOIN halls ON group_sessions.hall_id = halls.id
    INNER JOIN gyms ON halls.gym_id = gyms.id
    INNER JOIN session_bookings ON session_bookings.session_id = group_sessions.id

    -- One user cannot book the same group session twice, so it should be unique, but check
    WHERE session_bookings.user_id = ?;

    """, (user_id,))

visit_history = cur.fetchall()

# print("Visit history...")
# for v in visit_history:
#     print(v)

start_date = '2026-01-01'

def visit_history(user_id, start_date):
    '''Returns user history since a given start date'''

    cur.execute("""
    SELECT 
        activities.name, 
        gyms.gym_name, 
        group_sessions.sessiondate, 
        group_sessions.start_time
    FROM group_sessions
        INNER JOIN activities ON group_sessions.activity_id = activities.id
        INNER JOIN halls ON group_sessions.hall_id = halls.id
        INNER JOIN gyms ON halls.gym_id = gyms.id
        INNER JOIN session_bookings ON session_bookings.session_id = group_sessions.id

    -- One user cannot book the same group session twice, so it should be unique, but check
    WHERE 
        session_bookings.user_id = ?
        AND group_sessions.sessiondate >= ?
    ;

    """, (user_id, start_date))

    visit_history = cur.fetchall()

    return visit_history

visit_histories = visit_history(user_id, start_date)

for v in visit_histories:
    print(v)

('Spin 8x3min', 'Øya', '2026-03-16', '18:00')
('Spin60', 'Øya', '2026-03-17', '18:30')
('Spin 4x4', 'Dragvoll', '2026-03-17', '17:00')


## Use case 7: Who trained the most group sessions

# Test of program
Not done. I need to make it into a loop so that if a user have booked, is notified that they cannot book, etc., they are still logged in and can make other choices +++

In [12]:
import pandas as pd

query = """SELECT * FROM users"""

df = pd.read_sql_query(query, conn)

df

,id,first_name,last_name,email,phone_number,student
0,1,Johnny,Nordmann,johnny@stud.ntnu.no,40100001,1
1,2,Viola,Smith,viola@stud.ntnu.no,40100010,1
2,3,Erik,Hansen,erik.hansen@stud.ntnu.no,40100002,1
3,4,Sara,Berg,sara.berg@stud.ntnu.no,40100003,1
4,5,Lucas,Johansen,lucas.johansen@stud.ntnu.no,40100004,0
5,6,Emma,Larsen,emma.larsen@stud.ntnu.no,40100005,1
6,7,Noah,Dahl,noah.dahl@stud.ntnu.no,40100006,0
7,8,Maja,Nilsen,maja.nilsen@stud.ntnu.no,40100007,1
8,9,Oliver,Strand,oliver.strand@stud.ntnu.no,40100008,0
9,10,Ingrid,Moen,ingrid.moen@stud.ntnu.no,40100009,1


In [21]:
########## Imports ##########
import sqlite3
import datetime
from datetime import timedelta
import sys

########## DB Connection ##########

conn = sqlite3.connect("gym.db") # connect to database (this creates gym.db if it does not exist)


cur = conn.cursor() # cursor to execute SQL


########## Functions and procedures ##########

def check_dots_30_days(user_id):
    '''
    Checks how many dots were given the last 30 days and if user can train
    user_id : Users id 
    '''
    thirtydays = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
    cur.execute("""SELECT COUNT(*) FROM dot_system WHERE user_id=? AND date_given >= ?;""", (user_id, thirtydays))
    active_dots = cur.fetchone()[0]
    if active_dots >= 3:
        print("User is blacklisted.")
        return 0
    else:
        print("Booking allowed ...")
        return 1



def dot_expire(user_id):
    '''Function that returns oldest dot, useful to tell user when it expires'''
    cur.execute("""SELECT date_given FROM dot_system WHERE user_id=? ORDER BY date_given ASC LIMIT 1;""", (user_id,))  # Selects date of oldest dot
    date_given = cur.fetchone()[0] 

    date_given_dt = datetime.strptime(date_given, "%Y-%m-%d")  # Date dot was given
    dot_expires = date_given_dt + timedelta(days=30)  # Date dot expires
    dot_expires = dot_expires.date()
    today = datetime.now()  # Today
    today = today.date()
    return date_given_dt, dot_expires


#### Book session
from datetime import datetime


def book_session(user_id, session_id):
    '''
    Function that performs a booking
    --------------------
    user_id : Users unique id
    session_id: Sessions unique id (is retrieved from ****)
    '''

    #### First check if available spots by comparing booked spots with capacity
    cur.execute("""SELECT COUNT(*) FROM session_bookings WHERE session_id = ?;""", (session_id,))
    booked = cur.fetchone()[0]

    cur.execute("""SELECT capacity FROM halls JOIN group_sessions ON halls.id = group_sessions.hall_id WHERE group_sessions.id = ?;""", (session_id,))

    capacity = cur.fetchone()[0]

    if booked >= capacity:
        print("Session is full")
        return
    
    # TODO: Must register before booking deadline

    # If did not return above, means it was available spots so booking is performed:
    booking_time = str((datetime.now()).replace(microsecond=0))  # Booking time in format YYY-MM-DD HH:MM:SS

    cur.execute("""INSERT INTO session_bookings (session_id, user_id, booking_time) VALUES (?, ?, ?);""", (session_id, user_id, booking_time))
    booking_id = cur.lastrowid  # the autogenerated PK for that booking

    conn.commit()

    print("Booking successful")
    return booking_id


def arrived_in_time(arrival_time, session_time):
    
    '''
    Function that checks if time1 is no less than 5 minutes before time2, and converts between string and datetime

    Returns True if user arrived on time, False if not
    '''

    # Make the times, which is in string, into datetime objects
    arrival_dt = datetime.strptime(arrival_time, "%Y-%m-%d %H:%M:%S") 
    session_dt = datetime.strptime(session_time, "%Y-%m-%d %H:%M:%S")
    # dt_new = dt - timedelta(minutes=5) # subtract 5 minutes
    # return dt_new.strftime("%Y-%m-%d %H:%M:%S")  # Return 5 minutes before

    five_min_before = session_dt - timedelta(minutes=5)

    print(f"Start time was {session_dt}\nDeadline for showing up is {five_min_before}.\nUser arrived at {arrival_dt}")

    if arrival_dt <= five_min_before:
        print("User arrived in time!")
        return 1  # user arrived in time
    else:
        print("User did not arrive in time. A dot is created.")
        return 0


def dot_giver(user_id, booking_id, arrival_time, session_time):
    '''Function that gives a dot to user if user did not arrive in time'''
    if arrived_in_time(arrival_time, session_time):
        # True if arrived on time
        print("All good!")
        return
    else:
        
        # Make date for the dot
        today = datetime.now()
        date_part = today.date()
        today_str = str(date_part)

        # Insert dot in system
        cur.execute("""INSERT INTO dot_system (user_id, session_booking_id, date_given) VALUES (?, ?, ?)""", (user_id, booking_id, today_str))

        conn.commit()



def visit_history(user_id, start_date):
    '''
    Returns user history since a given start date
    
    user_id : Users' unique ID
    start_date : Which date should be shown, in format YYYY-MM-DD
    '''

    cur.execute("""
    SELECT 
        activities.name, 
        gyms.gym_name, 
        group_sessions.sessiondate, 
        group_sessions.start_time
    FROM group_sessions
        INNER JOIN activities ON group_sessions.activity_id = activities.id
        INNER JOIN halls ON group_sessions.hall_id = halls.id
        INNER JOIN gyms ON halls.gym_id = gyms.id
        INNER JOIN session_bookings ON session_bookings.session_id = group_sessions.id

    -- One user cannot book the same group session twice, so it should be unique, but check
    WHERE 
        session_bookings.user_id = ?
        AND group_sessions.sessiondate >= ?
    ;

    """, (user_id, start_date))

    visit_history = cur.fetchall()

    return visit_history



# def show_classes(week_num, start_day):
#     ''' Function that prints classes'''

#     week_num = None #fix later

#     cur.execute(
#         """SELECT 
#                 g.gym_name AS Gym,
#                 a.name,
#                 a.description,
#                 s.sessiondate AS Day,
#                 s.start_time AS Time
#                 --s.signup_deadline
#             FROM group_sessions s
#             INNER JOIN activities a ON a.id = s.activity_id
#             INNER JOIN halls h ON h.id = s.hall_id
#             INNER JOIN gyms g ON g.id = h.gym_id
#             WHERE s.sessiondate <= ?
#             -- AND s.week = ?
#             ORDER BY s.sessiondate, s.start_time
#             ;
#             """, (start_day,)
#             )
    
#     schedule = cur.fetchall()

#     rows = []

#     print(f"Schedule for week {week_num}, starting on {start_date}:")

#     for s in schedule:
#         row = [s[i] for i in range(len(s))]
#         rows.append(row)

#     for row in rows:
#         print(f"==============================\n{row[3]} {row[4]}\n    {row[0]}\n    {row[1]}\n------------------------------\n    {row[2]}\n==============================\n\n\n")

#     # return schedule TODO

def show_classes(week_num, start_day):
    ''' Function that prints classes'''

    week_num = None #fix later

    cur.execute(
        """SELECT 
                g.gym_name AS Gym,
                a.name,
                a.description,
                s.sessiondate AS Day,
                s.start_time AS Time
                --s.signup_deadline
            FROM group_sessions s
            INNER JOIN activities a ON a.id = s.activity_id
            INNER JOIN halls h ON h.id = s.hall_id
            INNER JOIN gyms g ON g.id = h.gym_id
            WHERE s.sessiondate > ?
            -- AND s.week = ?
            ORDER BY s.sessiondate, s.start_time
            ;
            """, (start_day,)
            )
    
    schedule = cur.fetchall()

    rows = []

    for s in schedule:
        row = [s[i] for i in range(len(s))]
        rows.append(row)

    # print(rows)
    for row in rows:
        print(f"==============================\n{row[3]} {row[4]}\n    {row[0]}\n    {row[1]}\n------------------------------\n    {row[2]}\n==============================\n\n\n")



########## User login ##########

email = str(input("Please enter username to log in: "))
cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
email_to_id = cur.fetchone()[0]  # user_id stored here
print(f"Welcome, {email}! user_id: {email_to_id}\n----------\n 1. Book session\n xx. Log out")

choice = int(input(f"Please choose what you want to do today from the list by entering the number on the left: "))

# if choice == 1:

########## Select session ##########

# TODO change to function that prints available sessions for week and start_date
start_day = input("Enter start day (YYYY-MM-DD): ")
show_classes(1, start_day)


cur.execute("""SELECT distinct sessiondate FROM group_sessions ORDER BY sessiondate""")

dates = cur.fetchall()

print("Please select date by entering the number on the left:")
for i, d in enumerate(dates):
    print(i + 1, d[0])  # count from 1

date = int(input("Choose date: "))
selected_date = dates[date - 1][0]  # To take only the date string from a list of tuples on the form [(), ()], counting from zero

print("Selected date: ", selected_date) 



########## Select training ##########


cur.execute("""SELECT g.id, a.name, g.start_time, a.description, g.signup_deadline FROM activities a INNER JOIN group_sessions g ON a.id = g.activity_id WHERE sessiondate=? ORDER BY g.start_time""", (selected_date,))

trainings = cur.fetchall()

print("Please select session by entering the number on the left:")
for i, (session_id, name, start_time, description, signup_deadline) in enumerate(trainings, start=1):
    print(f"{i}. {name}, starting {start_time}, signup deadline {signup_deadline}:")
    print(f"    {description}")
    print("     ")

training = int(input("Choose date: "))
selected_training = trainings[training - 1]  # To take only the training string from a list of tuples on the form [(), ()], counting from zero

print("Selected training: ", selected_training)

session_id, name, session_time, description, signup_deadline = selected_training

# print(name)
# print(session_id)

if check_dots_30_days(email_to_id):  # returns true if okay, false if three or more dots
    booking_id = book_session(email_to_id, session_id)
elif not check_dots_30_days(email_to_id):
    firstdot, expiration = dot_expire(email_to_id)
    print(f"You cannot make any bookings while you are blacklisted\nThe blacklisting lasts for as long as you have three dots in the system.\nWhen the first dot expires, you can book sessions again.\nThe first dot was given at {firstdot} and expires at {expiration}\nYou can begin booking from ...")
    
    # TODO make it instead go back to start, when loop is created later


print("Time is passing ... Your session is soon ....")

arrival_bin = int(input("Did you arrive do the session at least 5 minutes before? (1 for yes, 0 for no): "))

if arrival_bin == 0:   # TODO: Need to make it match the sessions time, instead of what I do now

    diff = -4
    session_time = (datetime.now()).replace(microsecond=0)  # Datetime now 
    arrived = session_time + timedelta(minutes=diff) # apply change

    # Make strings
    arrival_time = arrived.strftime("%Y-%m-%d %H:%M:%S")
    session_time = session_time.strftime("%Y-%m-%d %H:%M:%S")

    print(f"Bad! You arrived at {arrival_time} but session started at {session_time}")

elif arrival_bin == 1:
    diff = -5
    session_time = (datetime.now()).replace(microsecond=0)  # Datetime now 
    arrived = session_time + timedelta(minutes=diff) # apply change

    # Make strings
    arrival_time = arrived.strftime("%Y-%m-%d %H:%M:%S")
    session_time = session_time.strftime("%Y-%m-%d %H:%M:%S")

    print(arrival_time)
    print(session_time)

    print(f"Great! You arrived at {arrival_time} and session started at {session_time}")

# if not arrived_in_time(arrival_time, session_time):
    # returns 0 if user did not arrive in time

# dot_giver handles both checking if the time user arrived is good or not, and give dot if not
dot_giver(email_to_id, session_id, arrival_time, session_time)

visit_histories = visit_history(email_to_id, '2026-01-01')
# TODO automate date
print("Printing history..")
for v in visit_histories:
    print(v)



# elif choice == 0:
#     print("Bye!")

Welcome, sara.berg@stud.ntnu.no! user_id: 4
----------
 1. Book session
 xx. Log out
2026-03-17 17:00
    Dragvoll
    Spin 4x4
------------------------------
    Intervalltime på sykkel med god oppvarming, deretter 4 stående intervaller på 4 minutter med ca 2 minutter aktiv pause mellom hvert drag. Timen avsluttes med nedsykling.



2026-03-17 18:30
    Øya
    Spin60
------------------------------
    Spinningtime med variert løype fordelt på 2-4 arbeidsperioder. Litt raskere tempo og lengre stående perioder enn Spin45, men du bestemmer selv intensiteten med motstanden du legger på.



2026-03-18 16:00
    Dragvoll
    Spin45
------------------------------
    Spinningtime med variert løype fordelt på 2-3 arbeidsperioder som passer for alle. Du bestemmer selv intensiteten emd hvor mye motstand du legger på.



2026-03-18 19:00
    Øya
    Spin 4x4
------------------------------
    Intervalltime på sykkel med god oppvarming, deretter 4 stående intervaller på 4 minutter med ca 2 minut

## Version 2

### Correct version 2

In [ ]:
### Main program

########## Imports ##########
import sqlite3
import datetime
from datetime import timedelta
from datetime import datetime
import sys
import os

########## Functions and procedures ##########

def check_dots_30_days(user_id):
    '''
    Checks how many dots were given the last 30 days and if user can train
    user_id : Users id 
    '''
    thirtydays = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
    cur.execute("""SELECT COUNT(*) FROM dot_system WHERE user_id=? AND date_given >= ?;""", (user_id, thirtydays))
    active_dots = cur.fetchone()[0]
    if active_dots >= 3:
        print("User is blacklisted.")
        return 0
    else:
        print("Booking allowed ...")
        return 1



def dot_expire(user_id):
    '''Function that returns oldest dot, useful to tell user when it expires'''
    cur.execute("""SELECT date_given FROM dot_system WHERE user_id=? ORDER BY date_given ASC LIMIT 1;""", (user_id,))  # Selects date of oldest dot
    date_given = cur.fetchone()[0] 

    date_given_dt = datetime.strptime(date_given, "%Y-%m-%d")  # Date dot was given
    dot_expires = date_given_dt + timedelta(days=30)  # Date dot expires
    dot_expires = dot_expires.date()
    today = datetime.now()  # Today
    today = today.date()
    return date_given_dt, dot_expires




def book_session(user_id, session_id):
    '''
    Function that performs a booking
    --------------------
    user_id : Users unique id
    session_id: Sessions unique id (is retrieved from ****)
    '''

    #### First check if available spots by comparing booked spots with capacity
    cur.execute("""SELECT COUNT(*) FROM session_bookings WHERE session_id = ?;""", (session_id,))
    booked = cur.fetchone()[0]

    cur.execute("""SELECT capacity FROM halls JOIN group_sessions ON halls.id = group_sessions.hall_id WHERE group_sessions.id = ?;""", (session_id,))

    capacity = cur.fetchone()[0]

    if booked >= capacity:
        print("Session is full")
        return
    
    # TODO: Must register before booking deadline

    # If did not return above, means it was available spots so booking is performed:
    booking_time = str((datetime.now()).replace(microsecond=0))  # Booking time in format YYY-MM-DD HH:MM:SS

    cur.execute("""INSERT INTO session_bookings (session_id, user_id, booking_time) VALUES (?, ?, ?);""", (session_id, user_id, booking_time))
    booking_id = cur.lastrowid  # the autogenerated PK for that booking

    conn.commit()

    print("Booking successful")
    return booking_id


def arrived_in_time(arrival_time, session_time):
    
    '''
    Function that checks if time1 is no less than 5 minutes before time2, and converts between string and datetime

    Returns True if user arrived on time, False if not
    '''

    # Make the times, which is in string, into datetime objects
    arrival_dt = datetime.strptime(arrival_time, "%Y-%m-%d %H:%M:%S") 
    session_dt = datetime.strptime(session_time, "%Y-%m-%d %H:%M:%S")
    # dt_new = dt - timedelta(minutes=5) # subtract 5 minutes
    # return dt_new.strftime("%Y-%m-%d %H:%M:%S")  # Return 5 minutes before

    five_min_before = session_dt - timedelta(minutes=5)

    print(f"Start time was {session_dt}\nDeadline for showing up is {five_min_before}.\nUser arrived at {arrival_dt}")

    if arrival_dt <= five_min_before:
        print("User arrived in time!")
        return 1  # user arrived in time
    else:
        print("User did not arrive in time. A dot is created.")
        return 0


def dot_giver(user_id, booking_id, arrival_time, session_time):
    '''Function that gives a dot to user if user did not arrive in time'''
    if arrived_in_time(arrival_time, session_time):
        # True if arrived on time
        print("All good!")
        return
    else:
        
        # Make date for the dot
        today = datetime.now()
        date_part = today.date()
        today_str = str(date_part)

        # Insert dot in system
        cur.execute("""INSERT INTO dot_system (user_id, session_booking_id, date_given) VALUES (?, ?, ?)""", (user_id, booking_id, today_str))

        conn.commit()



def visit_history(user_id, start_date):
    '''
    Returns user history since a given start date
    
    user_id : Users' unique ID
    start_date : Which date should be shown, in format YYYY-MM-DD
    '''

    cur.execute("""
    SELECT 
        activities.name, 
        gyms.gym_name, 
        group_sessions.sessiondate, 
        group_sessions.start_time
    FROM group_sessions
        INNER JOIN activities ON group_sessions.activity_id = activities.id
        INNER JOIN halls ON group_sessions.hall_id = halls.id
        INNER JOIN gyms ON halls.gym_id = gyms.id
        INNER JOIN session_bookings ON session_bookings.session_id = group_sessions.id

    -- One user cannot book the same group session twice, so it should be unique, but check
    WHERE 
        session_bookings.user_id = ?
        AND group_sessions.sessiondate >= ?
    ;

    """, (user_id, start_date))

    visit_history = cur.fetchall()

    return visit_history


def show_classes(week_num, start_day):
    ''' Function that prints classes'''

    week_num = None #fix later

    cur.execute(
        """SELECT 
                g.gym_name AS Gym,
                a.name,
                a.description,
                s.sessiondate AS Day,
                s.start_time AS Time
                --s.signup_deadline
            FROM group_sessions s
            INNER JOIN activities a ON a.id = s.activity_id
            INNER JOIN halls h ON h.id = s.hall_id
            INNER JOIN gyms g ON g.id = h.gym_id
            WHERE s.sessiondate > ?
            -- AND s.week = ?
            ORDER BY s.sessiondate, s.start_time
            ;
            """, (start_day,)
            )
    
    schedule = cur.fetchall()

    rows = []

    for s in schedule:
        row = [s[i] for i in range(len(s))]
        rows.append(row)

    # print(rows)
    for row in rows:
        print(f"==============================\n{row[3]} {row[4]}\n    {row[0]}\n    {row[1]}\n------------------------------\n    {row[2]}\n==============================\n\n\n")


def is_db_initialized(cur):
    '''This function checks if tables have been created'''
    cur.execute("""SELECT name FROM sqlite_master WHERE type='table' AND name='gyms';""")
    return cur.fetchone() is not None


def login_menu(cur):
    '''
    This is the login menu for the user
    Returns user_id and email
    '''

    while True:
        email = str(input("Please enter username (your email) to log in, or type 'quit' to go back to main menu: "))

        if email.lower() == 'quit':
            return None  # go back to main

        cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
        result = cur.fetchone()  # user_id stored here

        if result is None:
            print("No user found with that email. Please try again")
            continue  # retry login

        # If login successful, it proceeds to this point
        print("Login successful.")
        user_id = result[0]
        return user_id, email
    
def user_menu(cur, user_id, email):
    '''Logged in user menu'''
    while True:
        print(f"Welcome, {email}! Your user_id is {user_id}\n------------------------------------------------------------\nYou are now in user menu. Please choose what you want to do by entering the commands (left) from the menu:\n    b : Book session\n    logout : Log out (go back to main menu)")

        user_input = str(input("> "))

        if user_input.strip().lower() == 'b':
            print("Let's book a group session!")
            #TODO
        elif user_input.strip().lower() == 'logout':
            print("Logging out... You will come back to main menu. To fully exit the program, please exit main menu as well.")
            return
        else:
            print("Not valid input. Please retry.")

def main_menu(cur, conn):
    '''Main menu after logging in'''
    while True:  # MAIN MENU

        print("------------------------------------------------------------\nYOU ARE NOW IN MAIN MENU\nChoose what you want to do by entering the command (left) and click enter:\n    e : Create tables and insert data (use case 1)\n    l : Log into system (use cases 2-6)\n    s : Statistics (use cases 7-8)\n    r : Reset database to start over\n    q : End the program.\n------------------------------------------------------------")

        user_input = str(input("> "))

        if user_input.strip().lower() == "q":
            print("Exiting program. Goodbye!")
            conn.close()  # Removes connection
            return

        elif user_input.strip().lower() == 'e':

            if is_db_initialized(cur):
                print("Database is already initialized. You must reset it (r) if you want to do this option again.")
            else:
                print("Creating tables and inserting data ...")

                ########## Connect to DB, create cursor, execute schema.sql and data.sql


                # Execute schema.sql
                with open("schema.sql", "r", encoding="utf-8") as f:  # Reads the file
                    schema_sql = f.read()
                cur.executescript(schema_sql)  # runs sql statements
                conn.commit()  # saves changes


                # Execute data.sql
                with open("data.sql", "r", encoding="utf-8") as f:
                    data_sql = f.read()
                cur.executescript(data_sql)
                conn.commit()

                # db_initialized = True  # Now this command can not be done again unless user resets db

                print("You may now use training-center-db. You can reset everything and start over by entering 'r'")

        elif user_input.strip().lower() == 'l':
            user_id, email = login_menu(cur)
            if user_id is not None:
                user_menu(cur, user_id, email)
            

def main():

    # db_initialized = False  # Database is not yet initialized

    conn = sqlite3.connect("gym.db") # connect to database (this creates gym.db if it does not exist)
    cur = conn.cursor() # cursor to execute SQL

    main_menu(cur, conn)  # Goes to main menu

main()

------------------------------------------------------------
YOU ARE NOW IN MAIN MENU
Choose what you want to do by entering the command (left) and click enter:
    e : Create tables and insert data (use case 1)
    l : Log into system (use cases 2-6)
    s : Statistics (use cases 7-8)
    r : Reset database to start over
    q : End the program.
------------------------------------------------------------


Database is already initialized. You must reset it (r) if you want to do this option again.
------------------------------------------------------------
YOU ARE NOW IN MAIN MENU
Choose what you want to do by entering the command (left) and click enter:
    e : Create tables and insert data (use case 1)
    l : Log into system (use cases 2-6)
    s : Statistics (use cases 7-8)
    r : Reset database to start over
    q : End the program.
------------------------------------------------------------
No user found with that email. Please try again
Login successful.
Welcome, johnny@stud.ntnu.no! Your user_id is 1
------------------------------------------------------------
You are now in user menu. Please choose what you want to do by entering the commands (left) from the menu:
    b : Book session
    logout : Log out (go back to main menu)
Logging out... You will come back to main menu.
------------------------------------------------------------
YOU ARE NOW IN MAIN MENU
Choose what you w

### Old version without main program

In [ ]:
########## Imports ##########
import sqlite3
import datetime
from datetime import timedelta
import sys

########## DB Connection ##########

conn = sqlite3.connect("gym.db") # connect to database (this creates gym.db if it does not exist)


cur = conn.cursor() # cursor to execute SQL


########## Functions and procedures ##########

def check_dots_30_days(user_id):
    '''
    Checks how many dots were given the last 30 days and if user can train
    user_id : Users id 
    '''
    thirtydays = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
    cur.execute("""SELECT COUNT(*) FROM dot_system WHERE user_id=? AND date_given >= ?;""", (user_id, thirtydays))
    active_dots = cur.fetchone()[0]
    if active_dots >= 3:
        print("User is blacklisted.")
        return 0
    else:
        print("Booking allowed ...")
        return 1



def dot_expire(user_id):
    '''Function that returns oldest dot, useful to tell user when it expires'''
    cur.execute("""SELECT date_given FROM dot_system WHERE user_id=? ORDER BY date_given ASC LIMIT 1;""", (user_id,))  # Selects date of oldest dot
    date_given = cur.fetchone()[0] 

    date_given_dt = datetime.strptime(date_given, "%Y-%m-%d")  # Date dot was given
    dot_expires = date_given_dt + timedelta(days=30)  # Date dot expires
    dot_expires = dot_expires.date()
    today = datetime.now()  # Today
    today = today.date()
    return date_given_dt, dot_expires


#### Book session
from datetime import datetime


def book_session(user_id, session_id):
    '''
    Function that performs a booking
    --------------------
    user_id : Users unique id
    session_id: Sessions unique id (is retrieved from ****)
    '''

    #### First check if available spots by comparing booked spots with capacity
    cur.execute("""SELECT COUNT(*) FROM session_bookings WHERE session_id = ?;""", (session_id,))
    booked = cur.fetchone()[0]

    cur.execute("""SELECT capacity FROM halls JOIN group_sessions ON halls.id = group_sessions.hall_id WHERE group_sessions.id = ?;""", (session_id,))

    capacity = cur.fetchone()[0]

    if booked >= capacity:
        print("Session is full")
        return
    
    # TODO: Must register before booking deadline

    # If did not return above, means it was available spots so booking is performed:
    booking_time = str((datetime.now()).replace(microsecond=0))  # Booking time in format YYY-MM-DD HH:MM:SS

    cur.execute("""INSERT INTO session_bookings (session_id, user_id, booking_time) VALUES (?, ?, ?);""", (session_id, user_id, booking_time))
    booking_id = cur.lastrowid  # the autogenerated PK for that booking

    conn.commit()

    print("Booking successful")
    return booking_id


def arrived_in_time(arrival_time, session_time):
    
    '''
    Function that checks if time1 is no less than 5 minutes before time2, and converts between string and datetime

    Returns True if user arrived on time, False if not
    '''

    # Make the times, which is in string, into datetime objects
    arrival_dt = datetime.strptime(arrival_time, "%Y-%m-%d %H:%M:%S") 
    session_dt = datetime.strptime(session_time, "%Y-%m-%d %H:%M:%S")
    # dt_new = dt - timedelta(minutes=5) # subtract 5 minutes
    # return dt_new.strftime("%Y-%m-%d %H:%M:%S")  # Return 5 minutes before

    five_min_before = session_dt - timedelta(minutes=5)

    print(f"Start time was {session_dt}\nDeadline for showing up is {five_min_before}.\nUser arrived at {arrival_dt}")

    if arrival_dt <= five_min_before:
        print("User arrived in time!")
        return 1  # user arrived in time
    else:
        print("User did not arrive in time. A dot is created.")
        return 0


def dot_giver(user_id, booking_id, arrival_time, session_time):
    '''Function that gives a dot to user if user did not arrive in time'''
    if arrived_in_time(arrival_time, session_time):
        # True if arrived on time
        print("All good!")
        return
    else:
        
        # Make date for the dot
        today = datetime.now()
        date_part = today.date()
        today_str = str(date_part)

        # Insert dot in system
        cur.execute("""INSERT INTO dot_system (user_id, session_booking_id, date_given) VALUES (?, ?, ?)""", (user_id, booking_id, today_str))

        conn.commit()



def visit_history(user_id, start_date):
    '''
    Returns user history since a given start date
    
    user_id : Users' unique ID
    start_date : Which date should be shown, in format YYYY-MM-DD
    '''

    cur.execute("""
    SELECT 
        activities.name, 
        gyms.gym_name, 
        group_sessions.sessiondate, 
        group_sessions.start_time
    FROM group_sessions
        INNER JOIN activities ON group_sessions.activity_id = activities.id
        INNER JOIN halls ON group_sessions.hall_id = halls.id
        INNER JOIN gyms ON halls.gym_id = gyms.id
        INNER JOIN session_bookings ON session_bookings.session_id = group_sessions.id

    -- One user cannot book the same group session twice, so it should be unique, but check
    WHERE 
        session_bookings.user_id = ?
        AND group_sessions.sessiondate >= ?
    ;

    """, (user_id, start_date))

    visit_history = cur.fetchall()

    return visit_history


def show_classes(week_num, start_day):
    ''' Function that prints classes'''

    week_num = None #fix later

    cur.execute(
        """SELECT 
                g.gym_name AS Gym,
                a.name,
                a.description,
                s.sessiondate AS Day,
                s.start_time AS Time
                --s.signup_deadline
            FROM group_sessions s
            INNER JOIN activities a ON a.id = s.activity_id
            INNER JOIN halls h ON h.id = s.hall_id
            INNER JOIN gyms g ON g.id = h.gym_id
            WHERE s.sessiondate > ?
            -- AND s.week = ?
            ORDER BY s.sessiondate, s.start_time
            ;
            """, (start_day,)
            )
    
    schedule = cur.fetchall()

    rows = []

    for s in schedule:
        row = [s[i] for i in range(len(s))]
        rows.append(row)

    # print(rows)
    for row in rows:
        print(f"==============================\n{row[3]} {row[4]}\n    {row[0]}\n    {row[1]}\n------------------------------\n    {row[2]}\n==============================\n\n\n")



########## User login ##########



email = str(input("Please enter username to log in: "))
cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
email_to_id = cur.fetchone()[0]  # user_id stored here
print(f"Welcome, {email}! user_id: {email_to_id}\n----------\n 1. Book session\n xx. Log out")

choice = int(input(f"Please choose what you want to do today from the list by entering the number on the left: "))

# if choice == 1:

########## Select session ##########

# TODO change to function that prints available sessions for week and start_date
start_day = input("Enter start day (YYYY-MM-DD): ")
show_classes(1, start_day)


cur.execute("""SELECT distinct sessiondate FROM group_sessions ORDER BY sessiondate""")

dates = cur.fetchall()

print("Please select date by entering the number on the left:")
for i, d in enumerate(dates):
    print(i + 1, d[0])  # count from 1

date = int(input("Choose date: "))
selected_date = dates[date - 1][0]  # To take only the date string from a list of tuples on the form [(), ()], counting from zero

print("Selected date: ", selected_date) 



########## Select training ##########


cur.execute("""SELECT g.id, a.name, g.start_time, a.description, g.signup_deadline FROM activities a INNER JOIN group_sessions g ON a.id = g.activity_id WHERE sessiondate=? ORDER BY g.start_time""", (selected_date,))

trainings = cur.fetchall()

print("Please select session by entering the number on the left:")
for i, (session_id, name, start_time, description, signup_deadline) in enumerate(trainings, start=1):
    print(f"{i}. {name}, starting {start_time}, signup deadline {signup_deadline}:")
    print(f"    {description}")
    print("     ")

training = int(input("Choose date: "))
selected_training = trainings[training - 1]  # To take only the training string from a list of tuples on the form [(), ()], counting from zero

print("Selected training: ", selected_training)

session_id, name, session_time, description, signup_deadline = selected_training

# print(name)
# print(session_id)

if check_dots_30_days(email_to_id):  # returns true if okay, false if three or more dots
    booking_id = book_session(email_to_id, session_id)
elif not check_dots_30_days(email_to_id):
    firstdot, expiration = dot_expire(email_to_id)
    print(f"You cannot make any bookings while you are blacklisted\nThe blacklisting lasts for as long as you have three dots in the system.\nWhen the first dot expires, you can book sessions again.\nThe first dot was given at {firstdot} and expires at {expiration}\nYou can begin booking from ...")
    
    # TODO make it instead go back to start, when loop is created later


print("Time is passing ... Your session is soon ....")

arrival_bin = int(input("Did you arrive do the session at least 5 minutes before? (1 for yes, 0 for no): "))

if arrival_bin == 0:   # TODO: Need to make it match the sessions time, instead of what I do now

    diff = -4
    session_time = (datetime.now()).replace(microsecond=0)  # Datetime now 
    arrived = session_time + timedelta(minutes=diff) # apply change

    # Make strings
    arrival_time = arrived.strftime("%Y-%m-%d %H:%M:%S")
    session_time = session_time.strftime("%Y-%m-%d %H:%M:%S")

    print(f"Bad! You arrived at {arrival_time} but session started at {session_time}")

elif arrival_bin == 1:
    diff = -5
    session_time = (datetime.now()).replace(microsecond=0)  # Datetime now 
    arrived = session_time + timedelta(minutes=diff) # apply change

    # Make strings
    arrival_time = arrived.strftime("%Y-%m-%d %H:%M:%S")
    session_time = session_time.strftime("%Y-%m-%d %H:%M:%S")

    print(arrival_time)
    print(session_time)

    print(f"Great! You arrived at {arrival_time} and session started at {session_time}")

# if not arrived_in_time(arrival_time, session_time):
    # returns 0 if user did not arrive in time

# dot_giver handles both checking if the time user arrived is good or not, and give dot if not
dot_giver(email_to_id, session_id, arrival_time, session_time)

visit_histories = visit_history(email_to_id, '2026-01-01')
# TODO automate date
print("Printing history..")
for v in visit_histories:
    print(v)



# elif choice == 0:
#     print("Bye!")

# Reset database during development for recreation

In [8]:
#### Reset database during development so that we can recreate it from scratch when needed

import os

conn.close()  # Must remove db connection first

if os.path.exists("gym.db"):
    os.remove("gym.db")